In [26]:
import pandas as pd
import numpy as np
from numba import njit
import matplotlib.pyplot as plt

In [2]:
def historical_data(filename,
                    start_date = 0,
                    end_date = 0,
                    drop = ['high', 'low', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'ignore']):
    data = pd.read_csv(filename, parse_dates=['timestamp'])
    data.set_index('timestamp', inplace=True)
    if start_date == 0:
        start_date = data.index[0]
    if end_date == 0:
        end_date = data.index[-1]
    data = data[start_date : end_date]
    data.drop(drop, axis=1, inplace=True)
    return data

In [56]:
@njit
def vectorized_sma(arr, window):

    out = np.empty(len(arr))
    out[:] = np.nan

    # Вычисляем накопленную сумму
    csum = np.cumsum(arr)

    # Вычисляем сумму по окну с помощью сдвига
    csum[window:] = csum[window:] - csum[:-window]

    # Заполняем результат, начиная с window-1
    out[window - 1:] = csum[window - 1:] / window

    return out

In [57]:
@njit
def fast_sma(arr, window):

    out = np.empty(len(arr))
    out[:] = np.nan

    csum = np.cumsum(arr)

    for i in range(window - 1, len(arr)):
        total = csum[i]

        if i >= window:
            total -= csum[i - window]

        out[i] = total / window

    return out

In [58]:
@njit
def signals(close_prices, short_window=50, long_window=200):

    # Рассчитываем SMA
    # sma50 = np.convolve(close_prices, np.ones(short_window)/short_window, mode='valid')
    # sma200 = np.convolve(close_prices, np.ones(long_window)/long_window, mode='valid')

    # sma50 = vectorized_sma(close_prices, short_window)
    # sma200 = vectorized_sma(close_prices, long_window)

    sma50 = fast_sma(close_prices, short_window)
    sma200 = fast_sma(close_prices, long_window)

    # Выравниваем длины массивов, добавляя NaN в начало
    # sma50 = np.concatenate((np.full(short_window - 1, np.nan), sma50))
    # sma200 = np.concatenate((np.full(long_window - 1, np.nan), sma200))

    # Сигналы для покупки
    buy_signal = np.where((sma50 >= sma200) & (np.roll(sma50, 1) <= np.roll(sma200, 1)), 1, 0)

    # Сигналы для продажи
    sell_signal = np.where((sma50 <= sma200) & (np.roll(sma50, 1) >= np.roll(sma200, 1)), -1, 0)

    # Итоговый сигнал: 1 (купить), -1 (продать), 0 (ничего не делать)
    signal = buy_signal + sell_signal

    return signal

In [59]:
@njit
def execute_strategy(open_price, signal,
                     initial_cash=100000.0,
                     buy_pct=0.8,
                     commission=0.001,
                     lot_size=0.0001):

    n = len(open_price)

    cash = initial_cash
    shares = 0

    equity = np.zeros(n)
    cash_arr = np.zeros(n)
    shares_arr = np.zeros(n)

    equity[0] = initial_cash
    cash_arr[0] = cash
    shares_arr[0] = 0

    for i in range(1, n):
        p = open_price[i]
        prev_signal = signal[i-1]

        # --- BUY ---
        if prev_signal == 1:

            # расчет позиции на сделку
            target_value = cash * buy_pct

            # перевод в лоты
            shares_to_buy = np.floor(target_value / p / lot_size) * lot_size

            cost = shares_to_buy * p
            fee = cost * commission

            if cost + fee <= cash and shares_to_buy > 0:
                cash -= (cost + fee)
                shares += shares_to_buy

        # --- SELL ---
        elif prev_signal == -1:
            shares_to_sell = shares

            if shares_to_sell > 0:
                proceeds = shares_to_sell * p
                fee = proceeds * commission

                cash += (proceeds - fee)
                shares = 0

        cash_arr[i] = cash
        shares_arr[i] = shares
        equity[i] = cash + shares * p

    return equity, cash, shares


In [60]:
# Перебор параметров SMA
@njit
def optimize_sma_strategy(open_prices,
                          close_prices,
                          short_min=5,
                          short_max=50,
                          long_min=20,
                          long_max=200):

    best_equity = -1.0

    best_short = 0
    best_long = 0

    results_count = 0

    # примерный размер массива результатов
    max_results = (short_max - short_min) * (long_max - long_min)

    results = np.zeros((max_results, 3))

    for short_window in range(short_min, short_max):

        for long_window in range(long_min, long_max):

            # short должен быть меньше long
            if short_window >= long_window:
                continue

            # =========================
            # Генерация сигналов
            # =========================
            signal = signals(
                close_prices,
                short_window,
                long_window
            )

            # =========================
            # Бэктест
            # =========================
            equity_curve, final_cash, final_shares = execute_strategy(
                open_prices,
                signal
            )

            final_equity = equity_curve[-1]

            # сохраняем результат
            results[results_count, 0] = short_window
            results[results_count, 1] = long_window
            results[results_count, 2] = final_equity

            results_count += 1

            # =========================
            # Лучшая стратегия
            # =========================
            if final_equity > best_equity:

                best_equity = final_equity
                best_short = short_window
                best_long = long_window

    return (
        best_short,
        best_long,
        best_equity,
        results[:results_count]
    )

In [65]:
# Загрузка исторических данных
name = "BTCUSDT_1h_10_years"
filename = f"data/1h_10_years/{name}.csv"
data = historical_data(filename, '2021-07-23', '2022-12-15')

In [66]:
open_prices = data['open'].values.astype(np.float64)
close_prices = data['close'].values.astype(np.float64)

best_short, best_long, best_equity, results = optimize_sma_strategy(
    open_prices,
    close_prices
)

print("BEST SHORT:", best_short)
print("BEST LONG:", best_long)
print("BEST EQUITY:", best_equity)

BEST SHORT: 43
BEST LONG: 49
BEST EQUITY: 79551.45912478986


In [43]:
def fast_sma(arr, window):

    out = np.empty(len(arr))
    out[:] = np.nan

    csum = np.cumsum(arr)

    for i in range(window - 1, len(arr)):
        total = csum[i]

        if i >= window:
            total -= csum[i - window]

        out[i] = total / window

    return out

def signals(close_prices, short_window=50, long_window=200):

    # Рассчитываем SMA
    # sma50 = np.convolve(close_prices, np.ones(short_window)/short_window, mode='valid')
    # sma200 = np.convolve(close_prices, np.ones(long_window)/long_window, mode='valid')

    sma50 = fast_sma(close_prices, short_window)
    sma200 = fast_sma(close_prices, long_window)

    # Выравниваем длины массивов, добавляя NaN в начало
    # sma50 = np.concatenate((np.full(short_window - 1, np.nan), sma50))
    # sma200 = np.concatenate((np.full(long_window - 1, np.nan), sma200))

    # Сигналы для покупки
    buy_signal = np.where((sma50 >= sma200) & (np.roll(sma50, 1) <= np.roll(sma200, 1)), 1, 0)

    # Сигналы для продажи
    sell_signal = np.where((sma50 <= sma200) & (np.roll(sma50, 1) >= np.roll(sma200, 1)), -1, 0)

    # Итоговый сигнал: 1 (купить), -1 (продать), 0 (ничего не делать)
    signal = buy_signal + sell_signal

    return signal

def execute_strategy(open_price, signal,
                     initial_cash=100000.0,
                     buy_pct=0.8,
                     commission=0.001,
                     lot_size=0.0001):

    n = len(open_price)

    cash = initial_cash
    shares = 0

    equity = np.zeros(n)
    cash_arr = np.zeros(n)
    shares_arr = np.zeros(n)

    equity[0] = initial_cash
    cash_arr[0] = cash
    shares_arr[0] = 0

    for i in range(1, n):
        p = open_price[i]
        prev_signal = signal[i-1]

        # --- BUY ---
        if prev_signal == 1:

            # расчет позиции на сделку
            target_value = cash * buy_pct

            # перевод в лоты
            shares_to_buy = np.floor(target_value / p / lot_size) * lot_size

            cost = shares_to_buy * p
            fee = cost * commission

            if cost + fee <= cash and shares_to_buy > 0:
                cash -= (cost + fee)
                shares += shares_to_buy

        # --- SELL ---
        elif prev_signal == -1:
            shares_to_sell = shares

            if shares_to_sell > 0:
                proceeds = shares_to_sell * p
                fee = proceeds * commission

                cash += (proceeds - fee)
                shares = 0

        cash_arr[i] = cash
        shares_arr[i] = shares
        equity[i] = cash + shares * p

    return equity, cash, shares

def optimize_sma_strategy(open_prices,
                          close_prices,
                          short_min=5,
                          short_max=50,
                          long_min=20,
                          long_max=200):

    best_equity = -1.0

    best_short = 0
    best_long = 0

    results_count = 0

    # примерный размер массива результатов
    max_results = (short_max - short_min) * (long_max - long_min)

    results = np.zeros((max_results, 3))

    for short_window in range(short_min, short_max):

        for long_window in range(long_min, long_max):

            # short должен быть меньше long
            if short_window >= long_window:
                continue

            # =========================
            # Генерация сигналов
            # =========================
            signal = signals(
                close_prices,
                short_window,
                long_window
            )

            # =========================
            # Бэктест
            # =========================
            equity_curve, final_cash, final_shares = execute_strategy(
                open_prices,
                signal
            )

            final_equity = equity_curve[-1]

            # сохраняем результат
            results[results_count, 0] = short_window
            results[results_count, 1] = long_window
            results[results_count, 2] = final_equity

            results_count += 1

            # =========================
            # Лучшая стратегия
            # =========================
            if final_equity > best_equity:

                best_equity = final_equity
                best_short = short_window
                best_long = long_window

    return (
        best_short,
        best_long,
        best_equity,
        results[:results_count]
    )

open_prices = data['open'].values.astype(np.float64)
close_prices = data['close'].values.astype(np.float64)

best_short, best_long, best_equity, results = optimize_sma_strategy(
    open_prices,
    close_prices
)

print("BEST SHORT:", best_short)
print("BEST LONG:", best_long)
print("BEST EQUITY:", best_equity)

BEST SHORT: 36
BEST LONG: 85
BEST EQUITY: 2332620.321482002
